# GUM Evaluation of a Frequency-Dependent Measurement Series

### `JCGM 100:2008` — *Evaluation of measurement data — Guide to the expression of uncertainty in measurement*

**Object of the analysis**

* 37 frequency settings between **10 kHz** and **200 MHz** (logarithmic 1-2-5 style sweep)
* **15 repeated readings** per frequency (runs 1 … 15)
* Measurand: the level `x`, reported in **dBm** (set `VALUE_UNIT` in the first code
  cell if your instrument uses a different unit)

**Deliverables of this notebook**

1. Complete GUM uncertainty budget per frequency (Type A + Type B → combined → expanded, `k = 2`).
2. A **whisker box plot for the coverage factor `k = 2`** (box = `x̄ ± u_A`, whiskers = `x̄ ± U`),
   plus the classical Tukey box plot of the raw sample distribution as a cross-check.
3. Statistical pre-tests that the GUM assumptions require: normality, outliers, run-to-run trend.
4. Rounding of the results according to GUM 8.1.3 / 5.3.2 (2 significant digits, 3 if the
   leading digit is 1 or 2).

In [ ]:
# ---------------------------------------------------------------------------
# Imports, configuration and global settings
# ---------------------------------------------------------------------------
from __future__ import annotations

import io
import math
import pathlib
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter, LogFormatterSciNotation, NullFormatter
import pandas as pd
from scipy import stats

warnings.simplefilter("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)

# ---------------------------------------------------------------------------
# Configuration -- ADAPT THESE TO YOUR SETUP
# ---------------------------------------------------------------------------
# Repository layout
DATA_DIR = pathlib.Path("data")
PROC_DIR = DATA_DIR / "processed"     # result tables written by this notebook
FIG_DIR = pathlib.Path("figures")     # plots written by this notebook

K = 2.0                 # requested coverage factor (GUM 6.5.3)
VALUE_UNIT = "dBm"      # unit of the measurand (level)
FREQ_UNIT = "MHz"       # unit of the internal frequency column (f_MHz)
# The frequency *axis and labels* are auto-formatted in engineering units
# (kHz below 1 MHz, MHz above) -- see fmt_freq() below.
ALPHA = 0.05            # significance level for the supporting statistical tests

# --- specification tolerance -------------------------------------------
# The specified tolerance is +/- 3 dB and it is expressed in dB.  It is applied
# to every single one of the 15 readings: a reading complies when its deviation
# from the mean of its frequency group is within +/- TOL_DB.
# It is NOT a limit on the measured level as a function of frequency, so it is
# never drawn as a band across the sweep - only as a per-reading envelope.
TOL_DB = 3.0

# Type B: resolution of the recorded values (last digit of the data = 0.01 dBm)
U_RESOLUTION = 0.01    # in the unit of the measurand
# Type B: further components as {name: (standard_uncertainty, degrees_of_freedom)}
OTHER_TYPE_B = {}       # e.g. {"probe_cal": (0.05, 50), "drift": (0.02, 9)}

# Apply the Grubbs outlier test (GUM 4.2.4 / ISO 17025: outliers must not be
# discarded without a documented criterion). Default: only flag, never delete.
DROP_GRUBBS_OUTLIERS = False

mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 160,
    "font.size": 9,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})
print(f"K = {K},  alpha = {ALPHA},  unit = {VALUE_UNIT!r}")

## 1. Raw data

The measurement series is **not embedded in this notebook**. It is read from

```
data/raw/measurement_series.tsv
```

a tab-separated file with one row per frequency setting and the columns
`f_MHz`, `Run 1 … Run 15` (37 rows × 15 runs). That file is the single source of truth:
there is no second copy inside the notebook, so the data on disk and the analysed data
cannot drift apart. Run the notebook **from the repository root** so the relative path
resolves.

> **Note on the duplicate frequency.** `f = 100 kHz` occurs **twice**, but with two
> completely different value ranges (~29 dBm vs. ~3 dBm). The two blocks are therefore
> **not repeated measurements of one and the same operating point** – they are two
> different measurement conditions (different range / probe / setup). They are kept as two
> separate sample groups `100 kHz (A)` and `100 kHz (B)`. This is statistically justified in
> section 4.1.

In [ ]:
# ---------------------------------------------------------------------------
# Input data: read from the repository, not embedded in the notebook
# ---------------------------------------------------------------------------
# data/raw/measurement_series.tsv holds the measurement series exactly as it was
# supplied: 37 frequency rows x 15 runs, tab separated, one row per frequency.
# It is the single source of truth -- the notebook never carries a second copy,
# so the file and the analysis can no longer drift apart.
RAW_FILE = DATA_DIR / "raw" / "measurement_series.tsv"
if not RAW_FILE.is_file():
    raise FileNotFoundError(
        f"{RAW_FILE} not found. Run the notebook from the repository root, or "
        f"restore the input with:  git checkout -- {RAW_FILE.as_posix()}"
    )

df = pd.read_csv(RAW_FILE, sep="\t")
if df.columns[0] != "f_MHz":
    raise ValueError(f"unexpected first column {df.columns[0]!r} in {RAW_FILE}")
df["f_MHz"] = df["f_MHz"].astype(float)
RUN_COLS = [c for c in df.columns if c.startswith("Run")]
_missing = [f"Run {i}" for i in range(1, 16) if f"Run {i}" not in RUN_COLS]
if _missing:
    raise ValueError(f"{RAW_FILE} is missing columns: {_missing}")

print(f"read {RAW_FILE}  ({len(df)} rows x {len(RUN_COLS)} runs)")

# Tag duplicated frequency points so they stay separable.
# A frequency that occurs more than once gets suffixed (A), (B), ... so that
# each block is evaluated as its own sample group.
def fmt_freq(f_mhz: float) -> str:
    """Readable engineering notation: 0.082 MHz -> '82 kHz', 200 MHz -> '200 MHz'."""
    return f"{f_mhz * 1e3:g} kHz" if f_mhz < 1.0 else f"{f_mhz:g} MHz"


freq_formatter = FuncFormatter(lambda v, _pos: fmt_freq(v) if v > 0 else "")


def split_at_duplicate(x_mhz, y) -> tuple[np.ndarray, np.ndarray]:
    """Break the connecting line at a frequency that was measured twice.

    100 kHz exists in two independent conditions, A and B, which are *not* two
    points of one curve.  The two branches therefore have to be joined the way
    the measurement actually went:

        ... 82 kHz --- 100 kHz (A)          low-frequency branch  -> (A)
        100 kHz (B) --- 135 kHz ...        high-frequency branch -> (B)

    A NaN is inserted *between* the two occurrences, so matplotlib draws two
    separate polylines and neither of them touches the other condition.
    """
    xs: list[float] = []
    ys: list[float] = []
    prev = None
    for f, v in zip(x_mhz, y):
        if prev is not None and f == prev:
            xs.append(np.nan)
            ys.append(np.nan)
        xs.append(float(f))
        ys.append(v)
        prev = f
    return np.asarray(xs), np.asarray(ys)

_dup = df["f_MHz"].value_counts()
_seen: dict[float, int] = {}
_labels = []
for f in df["f_MHz"]:
    _seen[f] = _seen.get(f, 0) + 1
    _labels.append(f"{fmt_freq(f)} ({chr(64 + _seen[f])})" if _dup[f] > 1
                   else fmt_freq(f))
df.insert(0, "group", _labels)

print(f"{len(df)} frequency groups, {df[RUN_COLS].shape[1]} runs each "
      f"-> {int(df[RUN_COLS].size)} raw values")
df.head()

In [ ]:
# Long ("tidy") table: one row per single reading. This is the working table
# for every statistic below.
long = df.melt(
    id_vars=["group", "f_MHz"],
    value_vars=RUN_COLS,
    var_name="run",
    value_name="x",
)
long["run"] = long["run"].str.extract(r"(\d+)").astype(int)
long = long.dropna(subset=["x"]).reset_index(drop=True)

print(f"{len(long)} readings |  duplicate frequency handled as groups: "
      f"{[g for g in df['group'] if '(' in g]}")
long.head(10)

## 2. The GUM model (JCGM 100:2008)

For a measurand estimated from $n$ independent repeated readings $x_i$ of equal weight:

| Step | GUM clause | Formula | Symbol used below |
|---|---|---|---|
| Best estimate | 4.1.1 | $\hat\mu = \bar x = \frac{1}{n}\sum_i x_i$ | `mean` |
| Experimental standard deviation | 5.4.1 | $s(x_i) = \sqrt{\frac{1}{n-1}\sum_i (x_i-\bar x)^2}$ | `s` |
| **Type A** standard uncertainty of the mean | 5.4.3 | $u_A(\bar x) = \dfrac{s(x_i)}{\sqrt n}$ | `u_A` |
| Type B component (resolution) | 4.3.7 | $u_{B} = \dfrac{\Delta}{\sqrt{12}}$ (rectangular) | `u_res` |
| Combined standard uncertainty | 5.1 | $u_c = \sqrt{u_A^2 + \sum_j u_{B,j}^2}$ | `u_c` |
| Effective degrees of freedom | Annex G | $\nu_{\mathrm{eff}} = \dfrac{u_c^4}{\sum_j u_j^4/\nu_j}$ | `nu_eff` |
| **Expanded** uncertainty | 6.2.2 | $U = k\,u_c$ | `U` |
| Coverage probability of that $U$ | 6.3 | $p \approx 2\,F_t\big(k;\nu_{\mathrm{eff}}\big)-1$ | `p_cov` |

**On the coverage factor `k = 2`.** GUM 6.5.2/6.5.3 associates `k ≈ 2` with a level of
confidence of *about 95 %* — this holds exactly only in the limit $\nu \to \infty$
(*normal* distribution). With $\nu_{\mathrm{eff}} = n-1 = 14$ the coverage is slightly
**below** 95 %. The notebook therefore reports, for every frequency:

* `U(k=2)` — the requested value, and
* `p_cov` — its **actual** coverage probability, plus
* `k95` — the coverage factor that would give exactly 95 % ($t_{0.975;\nu_{\mathrm{eff}}}$).

**Rounding (GUM 8.1.3, 5.3.2).** Results are rounded to two significant digits, or to
three if the leading digit is 1 or 2. `gum_round()` implements this rule.

In [ ]:
# ---------------------------------------------------------------------------
# GUM core routines
# ---------------------------------------------------------------------------
def gum_type_a(values, ddof: int = 1) -> dict:
    """GUM 5.4: mean, experimental standard deviation, Type A standard uncertainty."""
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    n = x.size
    mean = float(x.mean())
    s = float(x.std(ddof=ddof))
    u_a = s / math.sqrt(n)
    return dict(n=n, mean=mean, s=s, u_A=u_a, dof=n - 1,
                min=float(x.min()), max=float(x.max()), values=x)


def gum_round(value: float, sig: int | None = None) -> float:
    """GUM 8.1.3: 2 significant digits, 3 if the leading digit is 1 or 2."""
    if value == 0 or not np.isfinite(value):
        return value
    if sig is None:
        lead = int(f"{abs(value):.1e}"[0])
        sig = 3 if lead in (1, 2) else 2
    exp = math.floor(math.log10(abs(value)))
    return round(value, -(exp - sig + 1))


def coverage_probability(k: float, dof: float) -> float:
    """P(|T| < k) for T ~ Student-t(dof) -- GUM 6.3."""
    return float(2.0 * stats.t.cdf(abs(k), dof) - 1.0)


def coverage_factor(level: float = 0.95, dof: float = 14.0) -> float:
    """k that yields exactly `level` coverage -- GUM Annex G."""
    return float(stats.t.ppf(0.5 + level / 2.0, dof))


def welch_satterthwaite(components: list[tuple[str, float, float]]) -> dict:
    """GUM Annex G.4.  components = [(name, u_i, nu_i), ...]"""
    u_c = math.sqrt(sum(u ** 2 for _, u, _ in components))
    denom = sum(u ** 4 / nu for _, u, nu in components if np.isfinite(nu) and nu > 0)
    nu_eff = u_c ** 4 / denom if denom > 0 else float("inf")
    return dict(u_c=u_c, nu_eff=nu_eff, components=components)


# --- quick self-test of the two formulas ------------------------------------
_x = [1.0, 2.0, 3.0, 4.0]                                   # s = 1.29099, u_A = 0.6455
_t = gum_type_a(_x)
assert math.isclose(_t["s"], 1.2909944487, rel_tol=1e-9), _t["s"]
assert math.isclose(_t["u_A"], 0.6454972244, rel_tol=1e-9), _t["u_A"]
assert math.isclose(coverage_factor(0.95, 14), 2.1447866879, rel_tol=1e-9)
assert math.isclose(gum_round(0.012345), 0.0123, rel_tol=1e-12)   # leading digit 1 -> 3 digits
assert math.isclose(gum_round(0.045678), 0.046, rel_tol=1e-12)    # leading digit 4 -> 2 digits
print(f"self-test passed |  t(0.975;14) = {coverage_factor(0.95, 14):.4f}  |  "
      f"p(k=2, 14) = {coverage_probability(2.0, 14) * 100:.2f} %")

## 3. Type A evaluation — `u_A` of every frequency point

One row per frequency point. `s` is the *repeatability* of a single reading,
`u_A = s/√15` the standard uncertainty of the **mean value** (this is the quantity the
GUM uncertainty refers to — not the spread of the single readings!).

In [ ]:
type_a = long.groupby(["group", "f_MHz"], sort=False)["x"].agg(
    n="count", mean="mean", s="std", min="min", max="max"
).reset_index()
type_a["u_A"] = type_a["s"] / np.sqrt(type_a["n"])
type_a["U_k2"] = K * type_a["u_A"]
type_a["rel_U_%"] = 100.0 * type_a["U_k2"] / type_a["mean"].abs()
type_a["span"] = type_a["max"] - type_a["min"]
type_a["u_A_pct_of_span"] = 100.0 * type_a["u_A"] / type_a["span"]

type_a[["group", "n", "mean", "s", "u_A", "U_k2", "rel_U_%", "min", "max"]].round(4)

## 4. Statistical pre-tests required by the GUM

The GUM's Type A formulas rest on two assumptions:

1. **Normally distributed** readings — the Student-t coverage applies (GUM 6.3.3),
2. **independent and free of systematic drift** readings (GUM 4.1.2, 5.1).

Both are tested per frequency point:

* `Shapiro–Wilk` (normality, $H_0$: normal),
* `Grubbs` single-outlier test (GUM 4.2.4 — a documented, objective criterion is required
  before any reading may be discarded; the notebook only *flags*),
* **run trend** — two complementary indicators, because one linear slope alone is not a
  sufficient test:
  1. `drift_slope` / `drift_p` — OLS slope of the readings on the run index;
  2. `split_p` — Welch *t*-test of runs 1–7 against runs 8–15. This detects a monotone
     settling effect robustly and, unlike the linear regression, is not misled by a
     *non-monotone* transient (rise → overshoot → settle).

An indicator is a *symptom*, not proof. It shows where the assumption of independent,
identically distributed readings is questionable. Section 4.2 quantifies the consequence.

In [ ]:
def grubbs_test(values, alpha: float = 0.05) -> dict:
    """Grubbs' single-outlier test (two-sided). Returns the flagged index."""
    x = np.asarray(values, dtype=float)
    n = x.size
    s = x.std(ddof=1)
    dev = x - x.mean()
    idx = int(np.argmax(np.abs(dev)))
    G = np.abs(dev[idx]) / s
    t = stats.t.ppf(1.0 - alpha / (2.0 * n), n - 2)
    G_crit = (n - 1) / math.sqrt(n) * math.sqrt(t ** 2 / (n - 2 + t ** 2))
    return dict(G=G, G_crit=G_crit, run_index=idx, run=f"Run {int(np.argmax(np.abs(dev))) + 1}",
                outlier=bool(G > G_crit), value=float(x[idx]), n=n)


def split_half_test(values, alpha: float = 0.05) -> dict:
    """Welch t-test: first half of the runs vs. the remaining runs."""
    x = np.asarray(values, dtype=float)
    m = math.ceil(x.size / 2)
    a_, b_ = x[:m], x[m:]
    t, p = stats.ttest_ind(a_, b_, equal_var=False)
    return dict(first_half=float(a_.mean()), second_half=float(b_.mean()),
                diff=float(a_.mean() - b_.mean()), p=float(p), flagged=bool(p < alpha))


def trend_fit(values) -> dict:
    """OLS trend x = a + b*i over the run index and the scatter *about* that trend
    (GUM 4.1.2 and Annex H, correlated observations)."""
    x = np.asarray(values, dtype=float)
    n = x.size
    i = np.arange(1.0, n + 1.0)                      # run number 1..n
    b, a, *_ = stats.linregress(i, x)
    resid = x - (a + b * i)
    dof = n - 2
    s_res = float(np.sqrt(float((resid ** 2).sum()) / dof)) if dof > 0 else float("nan")
    return dict(i_bar=float(i.mean()), Sxx=float(((i - i.mean()) ** 2).sum()),
                slope=float(b), s_resid=s_res, u_A_detrended=s_res / math.sqrt(n))


rows = []
for g, sub in long.groupby("group", sort=False):
    a = gum_type_a(sub["x"].to_numpy())
    sh = stats.shapiro(a["values"])
    gr = grubbs_test(a["values"], ALPHA)
    slope, icpt, r, p, se = stats.linregress(sub["run"].to_numpy(), a["values"])
    sf = split_half_test(a["values"], ALPHA)
    tf = trend_fit(a["values"])
    rows.append(dict(
        group=g, n=a["n"], mean=a["mean"], s=a["s"],
        W_shapiro=sh.statistic, p_normality=float(sh.pvalue),
        normal_5pc=bool(sh.pvalue >= ALPHA),
        G_max=gr["G"], G_crit=gr["G_crit"], grubbs_outlier=gr["outlier"],
        grubbs_run=gr["run"], grubbs_run_no=int(gr["run_index"]) + 1,
        grubbs_value=gr["value"],
        drift_slope=slope, drift_slope_per_run_pct=100 * slope / abs(a["mean"]),
        drift_p=float(p), drift_significant=bool(p < ALPHA),
        split_diff=sf["diff"], split_p=sf["p"], split_significant=sf["flagged"],
        s_resid=tf["s_resid"], u_A_detrended=tf["u_A_detrended"],
        i_bar=tf["i_bar"], Sxx=tf["Sxx"],
    ))
tests = pd.DataFrame(rows)
tests["trend_detected"] = tests["drift_significant"] | tests["split_significant"]

print(f"Shapiro–Wilk: {int((~tests['normal_5pc']).sum())} / {len(tests)} groups deviate from "
      f"normality at {ALPHA:.0%}  (small n -> low power, only gross deviations show up)")
print(f"Grubbs:      {int(tests['grubbs_outlier'].sum())} / {len(tests)} groups contain a "
      f"statistically flagged outlier")
print(f"Run trend:   {int(tests['trend_detected'].sum())} / {len(tests)} groups show a "
      f"significant run-to-run trend (linear slope "
      f"{int(tests['drift_significant'].sum())}, split-half "
      f"{int(tests['split_significant'].sum())})")
print(f"mean |slope| = {tests['drift_slope'].abs().mean():.4f} {VALUE_UNIT}/run, "
      f"max |slope| = {tests['drift_slope'].abs().max():.4f} {VALUE_UNIT}/run "
      f"(= {tests['drift_slope'].abs().max() * 14:.2f} {VALUE_UNIT} over the 15 runs)")

_cols = ["group", "p_normality", "normal_5pc", "grubbs_outlier", "grubbs_run",
         "drift_slope_per_run_pct", "drift_p", "split_p", "trend_detected"]
tests.loc[tests["trend_detected"] | ~tests["normal_5pc"] | tests["grubbs_outlier"],
          _cols].round(5)

# ---------------------------------------------------------------------------
# Outlier handling.  GUM 4.2.4 does not permit discarding a reading without a
# documented, objective criterion; the Grubbs test provides one, but with
# n = 15 its power is low and several flags are marginal.  The default is
# therefore to FLAG ONLY.  Set DROP_GRUBBS_OUTLIERS = True at the top of the
# notebook and re-run this cell to see the effect on the budget.
# ---------------------------------------------------------------------------
if DROP_GRUBBS_OUTLIERS:
    flagged = [(r["group"], int(r["grubbs_run_no"])) for _, r in tests.iterrows()
               if r["grubbs_outlier"]]
    mask = long.set_index(["group", "run"]).index.isin(
        pd.MultiIndex.from_tuples(flagged, names=["group", "run"]))
    print(f"\nDROP_GRUBBS_OUTLIERS = True -> removing {int(mask.sum())} readings: "
          f"{', '.join(f'{g} run {r}' for g, r in flagged)}")
    long = long[~mask].reset_index(drop=True)
    print(f"{len(long)} readings remain - RE-RUN the following cells to propagate.")
else:
    n_flag = int(tests["grubbs_outlier"].sum())
    print(f"\nGrubbs flags in {n_flag} groups are REPORTED ONLY "
          f"(DROP_GRUBBS_OUTLIERS = False): no reading is discarded.")

### 4.1 Why `100 kHz` must be split into two groups

If the two `100 kHz` blocks were treated as *one* sample of $n=30$, the data would be
strongly **bimodal** and every GUM quantity would be meaningless: the mean would lie
*between* two physically different operating points, and $s$ would be inflated by a
systematic offset rather than by random noise.

The Welch *t*-test below shows that the two blocks are not two estimates of the same value.

In [ ]:
GROUP_A, GROUP_B = "100 kHz (A)", "100 kHz (B)"
a_block = long.loc[long["group"] == GROUP_A, "x"].to_numpy()
b_block = long.loc[long["group"] == GROUP_B, "x"].to_numpy()

t_stat, p_welch = stats.ttest_ind(a_block, b_block, equal_var=False)
lev_stat, p_levene = stats.levene(a_block, b_block, center="median")
_, p_shapiro_pooled = stats.shapiro(np.concatenate([a_block, b_block]))

print(f"group '{GROUP_A}': mean = {a_block.mean():.3f}  s = {a_block.std(ddof=1):.3f}  "
      f"u_A = {a_block.std(ddof=1)/math.sqrt(a_block.size):.4f}")
print(f"group '{GROUP_B}': mean = {b_block.mean():.3f}  s = {b_block.std(ddof=1):.3f}  "
      f"u_A = {b_block.std(ddof=1)/math.sqrt(b_block.size):.4f}")
print(f"\nWelch t-test      : t = {t_stat:.1f}, p = {p_welch:.3g}  "
      f"-> {'DIFFERENT' if p_welch < ALPHA else 'equal'} operating points")
print(f"Levene (variances): p = {p_levene:.3g}")
print(f"Shapiro on pooled : p = {p_shapiro_pooled:.3g}  "
      f"-> {'pooling is invalid (non-normal/bimodal)' if p_shapiro_pooled < ALPHA else 'pooling acceptable'}")
print(f"\n=> the two blocks are evaluated separately from here on.")

### 4.2 Trend-aware Type A evaluation (GUM 4.1.2, Annex H)

$u_A = s/\sqrt n$ is valid only if the readings are **uncorrelated** (GUM 4.1.2).
Where a run-to-run trend exists, GUM Annex H requires the standard deviation to be taken
about the *fitted trend* instead of about the mean:

$$s_{\text{resid}} = \sqrt{\frac{\sum_i \big[x_i-(a+b\,i)\big]^2}{n-2}},
\qquad u_{A,\text{resid}} = \frac{s_{\text{resid}}}{\sqrt n}.$$

With $i=1\ldots n$ symmetric around $\bar i$, the mean of the readings equals the trend
at $\bar i$, so the uncertainty of the mean is unchanged in form. Extrapolating the trend
to an arbitrary run index $r_0$ (e.g. to state the value *at the start* of a following
run) adds the term

$$u_A(r_0) = s_{\text{resid}}\sqrt{\frac1n + \frac{(r_0-\bar i)^2}{\sum_i (i-\bar i)^2}}.$$

The trade-off is fundamental and must be stated in any report:

* keeping $u_A=s/\sqrt n$ **inflates** the uncertainty with the trend,
* using $u_{A,\text{resid}}$ is correct for the *mean of these 15 runs*, but the estimated
  mean then sits at a drifting level — the *systematic* level shift must be carried as a
  Type B component, or eliminated by **randomising the run order**.

In [ ]:
# Reference run index the value is reported for:
#   8  -> the mean of the 15 runs (trend contribution vanishes)
#   1  -> the value at the start of the sequence (worst case for a linear trend)
R_REF = 8.0

drift = tests[["group", "n", "mean", "s", "s_resid", "i_bar", "Sxx",
               "drift_slope", "drift_p", "split_p", "trend_detected"]].copy()
drift["u_A_raw"] = drift["s"] / np.sqrt(15.0)
drift["u_A_resid"] = drift["s_resid"] / np.sqrt(15.0)
drift["u_A_at_R"] = drift["s_resid"] * np.sqrt(
    1.0 / 15.0 + (R_REF - drift["i_bar"]) ** 2 / drift["Sxx"])
drift["u_drift_B"] = drift["drift_slope"].abs() * abs(R_REF - drift["i_bar"]) / math.sqrt(3.0)
# equivalent Type B term if the value is required anywhere inside the 15-run
# interval (rectangular assumption for the unknown run index r0)
drift["u_drift_span"] = drift["drift_slope"].abs() * (drift["n"] - 1) / math.sqrt(3.0)
drift["ratio"] = drift["u_A_resid"] / drift["u_A_raw"]

print(f"Reference run index R_REF = {R_REF:g}  (i_bar = {drift['i_bar'].iloc[0]:g})")
print(f"across all groups: u_A(resid) / u_A(raw) = "
      f"{drift['ratio'].min():.2f} … {drift['ratio'].max():.2f}  "
      f"(median {drift['ratio'].median():.2f})")
print(f"groups where the trend inflates u_A by more than 25 %: "
      f"{int((drift['ratio'] < 0.8).sum())} / {len(drift)}")
print(f"Type B drift term at R_REF = {R_REF:g} (= mean run index): "
      f"{drift['u_drift_B'].max():.4f} {VALUE_UNIT}  "
      f"-> vanishing by construction, the mean is taken at the centre of the trend")
print(f"Type B drift term if the value may be required anywhere in the 15 runs: "
      f"{drift['u_drift_span'].min():.4f} … {drift['u_drift_span'].max():.4f} "
      f"{VALUE_UNIT}  -> this is the systematic effect that a non-randomised run "
      f"order introduces")
print(f"compare with the random Type A term u_A = s/sqrt(15): "
      f"{drift['u_A_raw'].min():.4f} … {drift['u_A_raw'].max():.4f} {VALUE_UNIT}")
print("\nStrongest trends (largest |slope| * (n-1)):")
drift.reindex(drift["drift_slope"].abs().sort_values(ascending=False).index).head(8)[
    ["group", "mean", "s", "s_resid", "u_A_raw", "u_A_resid", "ratio",
     "drift_slope", "trend_detected"]
].round(4)

In [ ]:
# ---------------------------------------------------------------------------
# Figure: trend diagnostic -- readings vs. run index for the strongest trends
# ---------------------------------------------------------------------------
top = drift.reindex(drift["drift_slope"].abs().sort_values(ascending=False).index).head(8)
fig5, ax5s = plt.subplots(2, 4, figsize=(12, 5.6), sharex=True)
for ax5, (_, r) in zip(ax5s.ravel(), top.iterrows()):
    v = long.loc[long["group"] == r["group"], "x"].to_numpy()
    i = np.arange(1, len(v) + 1)
    ax5.plot(i, v, "o-", color="0.35", ms=3.5, lw=0.9)
    ax5.axhline(v.mean(), color="tab:blue", lw=1.1)
    ax5.plot(i, r["mean"] + r["drift_slope"] * (i - 8.0), "-", color="tab:red", lw=1.1)
    ax5.set_title(f"{r['group']}\nslope {r['drift_slope']:+.3f} "
                  f"{VALUE_UNIT}/run,  p = {r['drift_p']:.1e}", fontsize=7.5)
    ax5.tick_params(labelsize=7)
for ax5 in ax5s[-1]:
    ax5.set_xlabel("run index  i")
for ax5 in ax5s[:, 0]:
    ax5.set_ylabel(f"x [{VALUE_UNIT}]")
fig5.suptitle("Run-to-run trend of the eight steepest groups "
              "(grey: the 15 readings, blue: mean, red: OLS slope)", fontsize=10)
fig5.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

## 5. Type B components, combined uncertainty and expanded uncertainty

With the raw file alone, only one Type B component can be justified:

**Resolution of the recording.** The values are given with a last digit of
$\Delta = 0.01\ \mathrm{dBm}$ → uniformly distributed half-width $\Delta/2$
(GUM 4.3.7, Table 2):  $u_{B} = \Delta/\sqrt{12} = 2.89\times10^{-3}\ \mathrm{dBm}$.

> **A GUM budget without the remaining Type B terms is incomplete.** Before this analysis
> is used in a report, add at least: calibration uncertainty of the reference standard,
> uncertainty of the frequency axis (the sweep points themselves are only given to 2–3
> digits!), probe/matching uncertainty, temperature drift, and the uncertainty of the
> *reading* of the instrument display. They are configured in `OTHER_TYPE_B` at the top of
> the notebook and automatically enter the budget.

Because the Type A part carries $\nu_A = n-1 = 14$ and the rectangular resolution term
carries (conservatively) $\nu \to \infty$, Welch–Satterthwaite gives $\nu_{\mathrm{eff}}$
just above 14; the Type B terms pull it up further, which is visible in the table.

Two budgets are computed for every frequency point and printed side by side:

| column | meaning |
|---|---|
| `u_A`, `U` | **trend-aware** Type A $s_{\text{resid}}/\sqrt n$ (GUM Annex H) wherever section 4.2 detected a run trend, otherwise the textbook $s/\sqrt n$ — this is the **recommended** result |
| `u_A_naive`, `U_naive` | plain GUM 5.4.3, $s/\sqrt n$ for every group — shown for comparison |

Both are combined with the *same* Type B components and the same `k`, so the two columns
differ only through the treatment of the run trend.

### 5.2 Specification tolerance: ±3 dB on each reading

The specified tolerance is **±3 dB**, and it is expressed in **dB**. It is applied to
**every single one of the 15 readings** of each frequency group: a reading complies when

$$|x_i - \bar{x}| \;\le\; TOL_{DB} \;=\; 3\ \mathrm{dB}.$$

It is **not** a limit on the measured level as a function of frequency, so it is never
drawn as a band across the sweep. On the two figures that show the readings, it appears as
a **±3 dB envelope around the mean of each frequency point** — a per-reading acceptance
criterion, not a passband.

**Result: all 555 readings comply.** The largest single deviation from a group mean is
`1.57 dB` at `100 kHz` block B, i.e. 48 % margin to the tolerance. The envelope is
therefore never the binding constraint here; what does constrain the result is the
uncertainty budget (section 5) and the non-stationarity of the measurand (section 4.2).

In [ ]:
# Type B components (identical for every frequency -> identical absolute contribution)
u_res = U_RESOLUTION / math.sqrt(12.0)
DF_RES = 1.0e9   # rectangular, no further knowledge -> treat as practically infinite dof

type_a = type_a.merge(
    tests[["group", "trend_detected", "s_resid", "drift_slope", "drift_p"]],
    on="group", how="left")
type_a["u_A_res"] = type_a["s_resid"] / np.sqrt(type_a["n"])
# Type A actually entering the budget: trend-aware (GUM Annex H) where a run trend
# is significant, the textbook s/sqrt(n) term everywhere else.
type_a["u_A_use"] = np.where(type_a["trend_detected"], type_a["u_A_res"], type_a["u_A"])


def build_budget(u_a_col: str) -> pd.DataFrame:
    """GUM 5.1: combine Type A and all Type B components, then expand (GUM 6.2.2)."""
    rows = []
    for _, r in type_a.iterrows():
        comps = [("A (mean of 15 runs)", r[u_a_col], r["n"] - 1),
                 ("B: resolution", u_res, DF_RES)]
        for name, (u_i, nu_i) in OTHER_TYPE_B.items():
            comps.append((f"B: {name}", float(u_i), float(nu_i)))
        ws = welch_satterthwaite(comps)
        u_c = ws["u_c"]
        U = K * u_c
        rows.append(dict(
            group=r["group"], f_MHz=r["f_MHz"], n=r["n"],
            mean=r["mean"], s=r["s"], min=r["min"], max=r["max"],
            u_A=r[u_a_col], u_res=u_res, u_c=u_c,
            nu_eff=ws["nu_eff"], U=U, p_cov=coverage_probability(K, ws["nu_eff"]),
            k95=coverage_factor(0.95, ws["nu_eff"]),
            rel_U_pct=100.0 * U / abs(r["mean"]),
            lo=r["mean"] - U, hi=r["mean"] + U,
        ))
    return pd.DataFrame(rows)


budget = build_budget("u_A_use")      # recommended: trend-aware where required
budget_naive = build_budget("u_A")    # textbook GUM 5.4.3, shown for comparison
budget["u_A_naive"] = budget_naive["u_A"].to_numpy()
budget["u_A_res"] = type_a["u_A_res"].to_numpy()
budget["U_naive"] = budget_naive["U"].to_numpy()
budget["trend_detected"] = type_a["trend_detected"].to_numpy()

# ---------------------------------------------------------------------------
# Specification tolerance: +/- TOL_DB on every single reading
# ---------------------------------------------------------------------------
# Applied to all 15 readings of every frequency group: a reading complies when
#   | x_i - mean(group) |  <=  TOL_DB
# The unit is dB.  This is a per-reading acceptance criterion, NOT a limit on
# the level versus frequency.
long = long.merge(budget[["group", "mean"]].rename(columns={"mean": "grp_mean"}),
                  on="group", how="left")
long["dev_dB"] = long["x"] - long["grp_mean"]
long["within_tol"] = long["dev_dB"].abs() <= TOL_DB

_g = long.groupby("group", sort=False).agg(
    n_ok=("within_tol", "sum"), max_abs_dev=("dev_dB", lambda s: s.abs().max()))
budget = budget.merge(_g, on="group", how="left")
budget["group_within_tol"] = budget["n_ok"] == budget["n"]

_bad = long[~long["within_tol"]]
print(f"\nTolerance: +/- {TOL_DB:g} dB on every single reading (n = {len(long)} readings)")
print(f"  readings within tolerance : {int(long['within_tol'].sum())} / {len(long)}")
print(f"  readings outside          : {len(_bad)}"
      + ("" if _bad.empty else " -> " + ", ".join(
          f"{r['group']} run {r['run']} ({r['dev_dB']:+.2f} dB)"
          for _, r in _bad.iterrows())))
_w = budget.loc[budget["max_abs_dev"].idxmax()]
print(f"  largest deviation from the mean: {budget['max_abs_dev'].max():.2f} dB "
      f"at {_w['group']}  -> margin to the tolerance: "
      f"{100 * (1 - _w['max_abs_dev'] / TOL_DB):.0f} %")
print(f"  all 37 groups fully within tolerance: {bool(budget['group_within_tol'].all())}")

C_TOL, C_TOL_EDGE = "#2ca02c", "#1a6b1a"


def draw_tolerance_envelope(ax, y, mean, lw: float = 7.0, alpha: float = 0.16,
                            label=None) -> None:
    """Draw the +/- TOL_DB acceptance envelope around each group's mean.

    This is drawn per frequency row, in dB, on the plots that show the 15
    readings.  It is deliberately NOT a band across the frequency axis.
    """
    ax.hlines(y, mean - TOL_DB, mean + TOL_DB, color=C_TOL, lw=lw, alpha=alpha,
              zorder=0, label=label)
    ax.hlines(y, mean - TOL_DB, mean - TOL_DB, color=C_TOL_EDGE, lw=0.9, alpha=0.75,
              zorder=0)
    ax.hlines(y, mean + TOL_DB, mean + TOL_DB, color=C_TOL_EDGE, lw=0.9, alpha=0.75,
              zorder=0)

print(f"Type B resolution:  u = {U_RESOLUTION:g}/sqrt(12) = {u_res:.5f} {VALUE_UNIT}  "
      f"(constant, dof -> inf)")
print(f"Type A share of the variance: "
      f"{100 * (budget['u_A'] ** 2 / budget['u_c'] ** 2).min():.1f} … "
      f"{100 * (budget['u_A'] ** 2 / budget['u_c'] ** 2).max():.1f} %")
print(f"nu_eff = {budget['nu_eff'].min():.1f} … {budget['nu_eff'].max():.1f}")
print(f"k = {K:g}  ->  actual coverage p = "
      f"{100 * budget['p_cov'].min():.2f} … {100 * budget['p_cov'].max():.2f} %   "
      f"(k for exactly 95 %: k95 = {budget['k95'].iloc[0]:.3f})")
print(f"trend-aware Type A used for {int(budget['trend_detected'].sum())} of "
      f"{len(budget)} groups -> U is smaller by up to "
      f"{100 * (1 - (budget['U'] / budget['U_naive']).min()):.0f} % there")
print("\n" + "=" * 118)
print("GUM uncertainty budget, k = 2   (values rounded according to GUM 8.1.3)")
print("=" * 118)
print(budget[["group", "mean", "u_A", "u_A_naive", "u_c", "nu_eff", "U", "U_naive",
              "rel_U_pct", "trend_detected"]]
      .assign(mean=lambda d: d["mean"].map(gum_round),
              u_A=lambda d: d["u_A"].map(gum_round),
              u_A_naive=lambda d: d["u_A_naive"].map(gum_round),
              u_c=lambda d: d["u_c"].map(gum_round),
              U=lambda d: d["U"].map(gum_round),
              U_naive=lambda d: d["U_naive"].map(gum_round),
              rel_U_pct=lambda d: d["rel_U_pct"].map(gum_round),
              nu_eff=lambda d: d["nu_eff"].round(1))
      .to_string(index=False))

### 5.1 Final result in GUM notation

$$\hat{\mu} = \bar{x} \pm U(k{=}2), \qquad U = 2\,\sqrt{u_A^2 + u_{B}^2}$$

i.e. the measurement result is reported as `mean (u_c)` or, as required by many
accreditation rules, as `mean ± U` together with the coverage factor and the corresponding
level of confidence. Since the measurand is a **dBm** level, note that uncertainties must
be added **linearly in dBm**, not in power — this is the correct GUM treatment, but it
means `U` is a dBm interval and not a linear-power interval.

In [ ]:
report = budget[["group", "f_MHz", "mean", "u_c", "U", "nu_eff", "k95", "p_cov",
                 "trend_detected", "rel_U_pct", "lo", "hi"]].copy()
report["rel_U_pct"] = report["rel_U_pct"].round(2)
report["p_cov"] = (100 * report["p_cov"]).round(2)
report["result"] = [f"{gum_round(m):g} {VALUE_UNIT} ± {gum_round(U):g} {VALUE_UNIT} (k = {K:g})"
                    for m, U in zip(report["mean"], report["U"])]
report[["group", "result", "rel_U_pct", "p_cov", "trend_detected"]].rename(
    columns={"p_cov": "coverage [%]", "trend_detected": "trend"}).reset_index(drop=True)

## 6. Whisker box plot for the coverage factor `k = 2`

This is the plot requested in the task. Every frequency point of the completed budget
(section 5) is drawn as a classical *box-and-whisker* glyph whose lengths are GUM
quantities:

* **centre (black dot)** — the best estimate $\hat\mu = \bar x$,
* **box** — $\bar x \pm u_A$: the standard uncertainty of the mean, i.e. `k = 1`,
* **whiskers with caps** — $\bar x \pm U$ with $U = k\,u_c$, `k = 2`; this is the
  **coverage interval** $\hat\mu \pm U$ of GUM 6.2.2,
* **thin grey line** — the observed range $\min \dots \max$ of the 15 readings, drawn
  for context only; it is *not* an uncertainty but the scatter of single readings.

All boxes are drawn as true rectangles in data coordinates, so their width is the real
size of $2u_A$ and can be compared directly with the $2U$ whiskers.

Because $U$ is roughly 30× smaller than the observed spread, the left panel alone makes
the GUM interval nearly invisible — which is itself the correct and important message.
The right panel therefore repeats the same glyphs **normalised to the group's own $u_A$**:
there the box always spans $\pm 1$, the whiskers $\pm U/u_A$, and the grey range shows how
many standard uncertainties of the mean the single readings actually scatter over.

In [ ]:
# ---------------------------------------------------------------------------
# Figure 1: GUM whisker box plot, k = 2
# ---------------------------------------------------------------------------
from matplotlib.patches import Rectangle

C_MEAN, C_BOX, C_WHIS, C_RANGE = "black", "tab:blue", "tab:red", "0.72"
BOX_H, CAP_H = 0.52, 0.26


def _draw_whiskers(ax, y, lo, hi, clo, chi, box_lo, box_hi, lw_box=1.2, lw_wh=1.8):
    """Classical box-and-whisker glyph, drawn in data coordinates.

    lo/hi      -> whisker ends (the horizontal whisker line)
    clo/chi    -> cap positions   (a short vertical cap at each end)
    box_lo/hi  -> the box spanning the standard uncertainty
    """
    ax.hlines(y, lo, hi, color=C_WHIS, lw=lw_wh, zorder=3)              # whiskers
    cap_y = np.concatenate([y - CAP_H, y + CAP_H, y - CAP_H, y + CAP_H])
    cap_x = np.concatenate([clo, clo, chi, chi])
    ax.hlines(cap_y, cap_x, cap_x, color=C_WHIS, lw=lw_wh, zorder=3)    # caps
    for yi, bl, bh in zip(y, box_lo, box_hi):                           # box
        ax.add_patch(Rectangle((bl, yi - BOX_H / 2), bh - bl, BOX_H,
                               facecolor=C_BOX, alpha=0.55, edgecolor=C_BOX,
                               lw=lw_box, zorder=4))


def plot_gum_whiskers(stat: pd.DataFrame, u_col: str = "U"):
    """Left panel: absolute scale.  Right panel: normalised to the group's own u_A."""
    d = stat.iloc[::-1].reset_index(drop=True)      # ascending frequency downwards
    y = np.arange(len(d))
    mean = d["mean"].to_numpy()
    u_a = d["u_A"].to_numpy()
    u = d[u_col].to_numpy()

    fig, (axA, axB) = plt.subplots(
        1, 2, figsize=(14.5, 10.5), sharey=True,
        gridspec_kw=dict(width_ratios=[1.35, 1.0], wspace=0.06))

    # ---------------- left: absolute scale
    draw_tolerance_envelope(axA, y, mean, label=rf"$\pm${TOL_DB:g} dB tolerance envelope")
    axA.hlines(y, d["min"], d["max"], color=C_RANGE, lw=1.4, zorder=1)
    _draw_whiskers(axA, y, mean - u, mean + u, mean - u, mean + u,
                   mean - u_a, mean + u_a)
    axA.plot(mean, y, "o", color=C_MEAN, ms=4.5, zorder=5)
    axA.set_xlim(0, max(d["max"]) * 1.05)
    axA.set_xlabel(f"measured value  x  [{VALUE_UNIT}]")
    axA.set_title("(a) absolute scale - GUM whisker box plot, k = %g\n"
                  "box = $\\bar{x}\\pm u_A$   |   whiskers = $\\bar{x}\\pm U$   |   "
                  "grey = min...max of the 15 readings" % K, fontsize=9.5)

    # ---------------- right: normalised to u_A
    z_lo = (d["min"].to_numpy() - mean) / u_a
    z_hi = (d["max"].to_numpy() - mean) / u_a
    z_u = u / u_a
    axB.hlines(y, z_lo, z_hi, color=C_RANGE, lw=1.4, zorder=1)
    _draw_whiskers(axB, y, -z_u, z_u, -z_u, z_u, -np.ones_like(y), np.ones_like(y))
    axB.plot(np.zeros_like(y), y, "o", color=C_MEAN, ms=4.5, zorder=5)
    axB.axvline(0, color=C_MEAN, lw=0.7, ls=":", zorder=2)
    for z in (-1, 1):
        axB.axvline(z, color=C_BOX, lw=0.6, ls=":", alpha=0.7, zorder=2)
    axB.set_xlabel(r"deviation from $\bar{x}$  in units of $u_A$")
    axB.set_title("(b) normalised to $u_A$\n"
                  r"box $=\pm1$   |   whiskers $=\pm U/u_A$   |   "
                  r"grey = observed min$\ldots$max", fontsize=9.5)
    lim = max(10.0, float(np.ceil(np.abs(np.concatenate([z_lo, z_hi, -z_u, z_u])).max())) + 1)
    axB.set_xlim(-lim, lim)
    step = 5 if lim <= 25 else 10
    axB.set_xticks(np.arange(-lim + 1, lim, step))

    for ax in (axA, axB):
        ax.set_yticks(y)
        ax.set_ylim(-1, len(d))
    axA.set_yticklabels(d["group"])
    axA.set_ylabel("frequency  f  (ascending downwards)")

    handles = [
        plt.Line2D([], [], color=C_MEAN, marker="o", ls="",
                   label=r"best estimate $\hat{\mu}=\bar{x}$"),
        plt.Line2D([], [], color=C_BOX, lw=8, alpha=0.55,
                   label=r"$\bar{x}\pm u_{A}$  (k = 1)"),
        plt.Line2D([], [], color=C_WHIS, lw=1.8,
                   label=r"$\bar{x}\pm U$  (coverage interval, k = %g)" % K),
        plt.Line2D([], [], color=C_RANGE, lw=1.4,
                   label="observed min…max of the 15 readings"),
        plt.Line2D([], [], color=C_TOL, lw=8, alpha=0.3,
                   label=rf"$\pm${TOL_DB:g} dB tolerance envelope"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=9,
               bbox_to_anchor=(0.5, 0.005))
    fig.suptitle("GUM evaluation of 15 readings per frequency point - "
                 "whisker box plot for the coverage factor k = %g" % K, fontsize=12)
    fig.tight_layout(rect=(0, 0.045, 1, 0.972))
    return fig


fig1 = plot_gum_whiskers(budget)
plt.show()

_ratio = budget["U"] / budget["u_A"]
_span = ((budget["max"] - budget["mean"]).abs() / budget["u_A"])
print(f"median U/u_A = {_ratio.median():.3f}  -> the k = 2 whiskers span "
      f"{2 * _ratio.median():.1f} u_A")
print(f"the observed min...max reaches {(_span.max()):.1f} u_A at most -> "
      f"the single readings scatter over roughly sqrt(15) = 3.87 times the width "
      f"of the coverage interval")

## 7. Cross-check: GUM interval vs. classical Tukey box plot

The classical box plot shows the distribution of the **single readings**; the GUM
uncertainty describes the **mean**. With $n=15$ the two must not be confused:

* Tukey: about 95 % of the *readings* lie inside `mean ± 1.725·s` (the 1.5·IQR fences);
* GUM $U(k{=}2) = 2\,s/\sqrt{15} = 0.516\,s$ — a factor $\sqrt{15}=3.87$ **smaller**.

The overlay confirms that the coverage intervals $\hat\mu \pm U$ sit well inside the
observed spread — they are not supposed to cover the individual readings, only the value
of the measurand.

In [ ]:
fig2, ax2 = plt.subplots(figsize=(11.5, 9.5))
groups = type_a["group"].tolist()
data = [long.loc[long["group"] == g, "x"].to_numpy() for g in groups]

bp = ax2.boxplot(data, vert=False, positions=np.arange(len(groups)), widths=0.55,
                 patch_artist=True, showfliers=True, whis=1.5,
                 flierprops=dict(marker=".", ms=5, mfc="0.4", mec="0.4", alpha=0.8),
                 medianprops=dict(color="0.2", lw=1.2),
                 whiskerprops=dict(color="0.45"), capprops=dict(color="0.45"))
for patch in bp["boxes"]:
    patch.set_facecolor("0.80")
    patch.set_edgecolor("0.45")

m = type_a["mean"].to_numpy()
U = budget["U"].to_numpy()
draw_tolerance_envelope(ax2, np.arange(len(groups)), m, label=rf"$\pm${TOL_DB:g} dB tolerance")
ax2.hlines(np.arange(len(groups)) - 0.30, m - U, m + U, color="tab:red", lw=2.0, zorder=5)
ax2.plot(m, np.arange(len(groups)) + 0.30, "k|", ms=14, mew=1.4, zorder=5)

ax2.set_yticks(np.arange(len(groups)))
ax2.set_yticklabels(groups)
ax2.invert_yaxis()
ax2.set_xlim(0, max(type_a["max"]) * 1.05)
ax2.set_xlabel(f"measured value  x  [{VALUE_UNIT}]")
ax2.set_ylabel("frequency  f")
ax2.set_title("Tukey box plot of the 15 single readings (grey) with the GUM result "
              "mean ± U(k=2) (red bar)\n"
              "U describes the MEAN, the box describes the SPREAD of the readings", fontsize=10)
ax2.legend(handles=[
    plt.Line2D([], [], color="0.55", lw=8, label="IQR box, 1.5·IQR whiskers (spread of readings)"),
    plt.Line2D([], [], color="tab:red", lw=2, label=r"$\bar{x}\pm U$  (k = 2, coverage of the mean)"),
    plt.Line2D([], [], color=C_TOL, lw=8, alpha=0.3,
               label=rf"$\pm${TOL_DB:g} dB tolerance envelope"),
], loc="lower right", fontsize=8)
fig2.tight_layout()
plt.show()

## 8. Response curve and frequency dependence of the uncertainty

Upper panel: mean value with the coverage band $\hat\mu \pm U$ (`k = 2`).
Lower panel: the two standard deviations on a double-logarithmic scale —

$$s(x_i) \propto \text{absolute repeatability}, \qquad u_A = s/\sqrt{15} \propto s/\sqrt{15}.$$

The constant vertical offset of $\sqrt{15} = 3.87$ between the two curves is the entire
benefit of the 15 repetitions: it is what the whole measurement programme buys.

In [ ]:
# sort by frequency, and by label within a frequency so that (A) is always
# placed before (B) -> the low-frequency branch joins (A), the high-frequency
# branch joins (B)
d = budget.sort_values(["f_MHz", "group"])
fx, fy = split_at_duplicate(d["f_MHz"], d["mean"])
_, flo = split_at_duplicate(d["f_MHz"], d["lo"])
_, fhi = split_at_duplicate(d["f_MHz"], d["hi"])

fig3, (axA, axB) = plt.subplots(2, 1, figsize=(9.5, 8.6), sharex=True,
                                 gridspec_kw=dict(height_ratios=[2.1, 1]))

# --- upper panel: response curve with the GUM coverage interval -------------
axA.fill_between(fx, flo, fhi, color="tab:red", alpha=0.25, lw=0, zorder=1)
axA.plot(fx, fy, "-", color="k", lw=1.2, zorder=3, label=r"mean $\bar{x}$")
axA.errorbar(d["f_MHz"], d["mean"], yerr=d["U"], fmt="none", ecolor="tab:red",
             elinewidth=1.1, capsize=3, capthick=1.1, zorder=2,
             label=r"coverage interval $\hat{\mu}\pm U$  (k = %g)" % K)
axA.plot(d["f_MHz"], d["mean"], "o", color="k", ms=3.5, zorder=4)


dup = d["f_MHz"].duplicated(keep=False)
if dup.any():
    for _, r in d[dup].iterrows():
        axA.annotate(r["group"], (r["f_MHz"], r["mean"]), textcoords="offset points",
                     xytext=(9, 0), fontsize=7.5, color="0.35", va="center")
    axA.annotate("two different conditions\nat the same frequency",
                 xy=(0.1, 16), xytext=(0.32, 21), fontsize=8, color="0.4",
                 arrowprops=dict(arrowstyle="->", color="0.5", lw=0.9))

axA.set_xscale("log")
axA.xaxis.set_major_formatter(freq_formatter)
axA.xaxis.set_minor_formatter(NullFormatter())
axA.set_ylabel(f"measured value  x  [{VALUE_UNIT}]")
axA.set_title("Frequency response with GUM coverage interval (k = %g)\n"
              "the coverage bars are barely visible - that is the honest scale of "
              "the uncertainty;\n"
              "at 100 kHz the low-frequency branch joins (A) and the high-frequency "
              "branch joins (B) - no line links the two conditions" % K, fontsize=9.5)
axA.legend(loc="upper right", fontsize=8)

# --- lower panel: where the uncertainty comes from --------------------------
# same break at 100 kHz as above: the two conditions are not one curve
_, y_s = split_at_duplicate(d["f_MHz"], d["s"])
_, y_ua = split_at_duplicate(d["f_MHz"], d["u_A"])
_, y_U = split_at_duplicate(d["f_MHz"], d["U"])

axB.loglog(fx, y_s, "-o", color="tab:green", ms=3.5,
           label=r"standard deviation of a single reading  $s(x_i)$")
axB.loglog(fx, y_ua, "-s", color="tab:blue", ms=3.5,
           label=r"standard uncertainty of the mean  $u_A$")
axB.loglog(fx, y_U, "-^", color="tab:red", ms=3.5,
           label=r"expanded uncertainty  $U = 2\,u_c$")
axB.yaxis.set_major_formatter(LogFormatterSciNotation())
axB.xaxis.set_major_formatter(freq_formatter)
axB.xaxis.set_minor_formatter(NullFormatter())
axB.set_ylabel(f"uncertainty  [{VALUE_UNIT}]")
axB.set_xlabel("frequency  f")
axB.legend(loc="upper right", fontsize=8, ncol=1)
axB.set_title(r"$\sqrt{15} = 3.87$: the gain obtained from 15 repetitions "
              r"(vertical distance $s\leftrightarrow u_A$)", fontsize=9)
fig3.tight_layout()
plt.show()

## 9. Relative expanded uncertainty over the sweep

The absolute uncertainty is roughly constant over large parts of the sweep
(~0.02 dBm), so the **relative** expanded uncertainty is strongly frequency dependent:
large where the measurand is small (here around `100 – 200 kHz`, where the measured
level drops to ~3–4 dBm), small on the plateau near 1 MHz and at high frequency.

This is the region where the Type B components (which are *absolute*) will dominate and
must therefore be quantified carefully. The ±3 dB tolerance of section 5.2 is a limit on
each *reading*, not on this relative quantity, so it is not drawn here.

In [ ]:
fig4, ax4 = plt.subplots(figsize=(9.5, 4.6))

# break the connecting line at 100 kHz: the branch below joins (A), the branch
# above joins (B), and no line is drawn between the two conditions
d4 = budget.sort_values(["f_MHz", "group"]).reset_index(drop=True)
y4 = d4["rel_U_pct"].to_numpy()
x4, y4_line = split_at_duplicate(d4["f_MHz"], y4)

ax4.semilogx(x4, y4_line, "-", color="tab:purple", lw=1.3)
ax4.semilogx(d4["f_MHz"], y4, "o", color="tab:purple", ms=4.5)

for _, r in budget.iterrows():
    if r["rel_U_pct"] > budget["rel_U_pct"].quantile(0.85):
        ax4.annotate(f"{r['group']}  {r['rel_U_pct']:.2f} %",
                     (r["f_MHz"], r["rel_U_pct"]), textcoords="offset points",
                     xytext=(4, 5), fontsize=7, color="0.3")
ax4.set_ylim(0, max(y4) * 1.18)
ax4.xaxis.set_major_formatter(freq_formatter)
ax4.xaxis.set_minor_formatter(NullFormatter())
ax4.set_xlabel("frequency  f")
ax4.set_ylabel(r"relative expanded uncertainty  $U/\hat{\mu}$  [%]")
ax4.set_title("Relative expanded uncertainty  $U/\\hat{\\mu}$  vs. frequency, k = %g\n"
              "a high relative uncertainty here is caused by a small level, not by a "
              "large absolute one;\n"
              "at 100 kHz the line is broken between the two measurement conditions"
              % K, fontsize=9.5)
fig4.tight_layout()
plt.show()

print(f"median relative expanded uncertainty: {budget['rel_U_pct'].median():.2f} %")
print(f"worst case: {budget.loc[budget['rel_U_pct'].idxmax(), 'group']} "
      f"with {budget['rel_U_pct'].max():.2f} %")
print(f"best case : {budget.loc[budget['rel_U_pct'].idxmin(), 'group']} "
      f"with {budget['rel_U_pct'].min():.2f} %")

In [ ]:
# ---------------------------------------------------------------------------
# Export
# ---------------------------------------------------------------------------
for d in (PROC_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# tidy copy of the input, with the per-reading deviation and tolerance verdict
long.sort_values(["group", "run"]).to_csv(
    PROC_DIR / "measurement_series_long.csv", index=False, float_format="%.6f")

for table, name in ((budget, "GUM_uncertainty_budget.csv"),
                    (type_a, "TypeA_statistics.csv"),
                    (tests, "Statistical_tests.csv"),
                    (drift, "Run_trend_analysis.csv")):
    table.to_csv(PROC_DIR / name, index=False, float_format="%.6f")

for f, n in ((fig1, "GUM_whisker_boxplot_k2.png"),
             (fig2, "Tukey_with_GUM_interval.png"),
             (fig3, "Response_with_coverage_band.png"),
             (fig4, "Relative_expanded_uncertainty.png"),
             (fig5, "Run_trend_diagnostic.png")):
    f.savefig(FIG_DIR / n, bbox_inches="tight")
    plt.close(f)

written = sorted(list(PROC_DIR.iterdir()) + list(FIG_DIR.iterdir()) +
                 [(DATA_DIR / "raw" / "measurement_series_long.csv")])
print("written:")
for f_ in written:
    print(f"  {str(f_):46s} {f_.stat().st_size/1024:8.1f} kB")

## 10. Result and discussion

**Statistical evaluation (GUM).** All 37 frequency points (555 readings) were evaluated:

$$\bar x=\tfrac1{15}\sum x_i,\qquad s=\sqrt{\tfrac1{14}\sum(x_i-\bar x)^2},\qquad
u_A=\tfrac{s}{\sqrt{15}},\qquad u_c=\sqrt{u_A^2+u_B^2},\qquad U = 2\,u_c .$$

**Findings.**

1. **The absolute repeatability spans a factor of 30.** $s$ ranges from `0.030` dBm
   (`1.8 MHz`) up to `0.94` dBm (the `100 kHz` block B); $U(k{=}2)$ consequently ranges from
   `0.017` dBm to `0.22` dBm (block B). The scatter is largest in the
   low-frequency transition region `10 – 200 kHz` and at the flanks of the resonance,
   smallest on the flat plateau between `1` and `3 MHz`.
2. **15 repetitions buy a factor of $\sqrt{15}=3.87$.** The mean value is determined almost
   four times more precisely than a single reading — and more than that if the runs are
   averaged in the frequency domain (VECTRAN / synchronous detection), which should be
   checked experimentally.
3. **The relative expanded uncertainty is worst where the level is low**: median `0.39 %`,
   worst case `6.7 %` at `100 kHz` (block B), best case `0.10 %` at `1.8 MHz`. In the
   low-level region the budget — not the statistics — decides the result, so the missing
   Type B terms of section 5 matter most exactly there.
4. **`k = 2` gives only ≈ 93.5 % coverage, not 95 %.** With $\nu_{\mathrm{eff}} \approx 14$
   the coverage of $2u_c$ is 93.5 %; for a strictly 95 % statement use
   $k = t_{0.975;14} = 2.144$ (≈ 7 % larger $U$). This matters only when a limit value has
   to be judged close to the boundary.
5. **The measurand was not stationary during 19 of the 37 series.** Both a linear slope
   (19/37) and a first-half/second-half comparison (18/37) reject the hypothesis of
   independent, identically distributed readings. The effect is strongest exactly where the
   response changes most: the `100 kHz` block B, `135 kHz` and `182 kHz` rise monotonically from run 1
   to run 15 (up to `+2.6` dBm over the series), and the `10 – 82 kHz` block shows a
   *non-monotone* transient (rise → overshoot → settle) which a linear test alone would
   misread. **This is the most important result of the analysis:** it is not a random error
   but a non-constant measurand, which limits the validity of any single mean value for
   those groups (GUM 3.1.9: the measurand must be well defined and constant over the
   observation interval). The trend-aware treatment of GUM Annex H reduces $U$ by up to
   57 % there — but that reduction is *not* a free win: the equivalent Type B drift term
   $|b|(n-1)/\sqrt3$ reaches `1.52` dBm at `100 kHz` block B, i.e. **six times** the random Type A
   term of `0.24` dBm at the same frequency. Removing the trend statistically therefore only
   hides the problem; the physically correct remedies are a longer settling time, a
   **randomised run order**, or an explicitly stated reference condition with the drift
   carried as a Type B component.
6. **The duplicate `100 kHz` entry is not a repetition.** The two blocks differ by a factor of
   ≈ 9 (Welch $t = 98$, $p \approx 10^{-26}$, Levene $p = 0.011$); pooling them gives a
   bimodal sample ($p_{\text{Shapiro}} < 10^{-6}$) whose mean lies between two physically
   different operating points. They are therefore evaluated as the separate groups
   `100 kHz (A)` and `100 kHz (B)`.
7. **12 of 37 groups deviate from normality** at the 5 % level. With $n = 15$ the
   Shapiro–Wilk test has limited power, and a significant result here is usually the
   fingerprint of finding 5 (a trend makes the sample non-normal) rather than heavy tails —
   normality is not violated in the groups with no trend.
8. **No reading was discarded.** Grubbs' test flags at least one reading in 9 of 37 groups,
   but with $n=15$ its power is low; the flags are reported and the readings are left in the
   data set (`DROP_GRUBBS_OUTLIERS = False`).
9. **All 555 readings comply with the ±3 dB tolerance** (section 5.2): the largest
   deviation of any single reading from its group mean is `1.57 dB`, at `100 kHz` block
   B, which leaves 48 % margin. The tolerance is therefore never the binding constraint
   in this data set.
10. **The relative expanded uncertainty is nevertheless strongly peaked** — median
    `0.39 %` but `6.66 %` at `100 kHz` block B and `4.08 %` at `135 kHz`. Both sit in the
    low-level region where the measurand is only 3–4 dBm, so an absolute uncertainty of
    ≈0.2 dBm becomes a large relative one. Relative uncertainty is the figure to watch
    when the level is small, even when the absolute numbers are comfortable.

**Open items before the result may be reported (GUM 4.3.1 / 5.3 completeness).**

* Add the missing Type B components — in particular the **calibration uncertainty of the
  reference** and the **uncertainty of the frequency axis**: the sweep points are only
  quoted to two or three digits, which for `1 MHz` already means ±50 kHz, and which
  for the steep flanks (`100 – 200 kHz`, ~2 MHz per 10 kHz step) dominates the horizontal
  axis error entirely.
* Specify how the 15 runs are combined in the instrument. If the instrument performs the
  averaging itself, the Type A analysis of the *displayed* values already contains that
  gain; if it displays single readings, $s/\sqrt{15}$ applies as computed here.
* Quantify the drift as a Type B component instead of removing it statistically
  (section 4.2): with a non-randomised run order the trend is a reproducible systematic
  effect and must not be absorbed into the random Type A budget.
* Repeat the sweep in **reverse frequency order** and, ideally, with a randomised run
  order, to separate hysteresis from settling drift.
* Repeat each series after a longer settling delay to verify whether finding 5 is a
  settling transient or a genuine drift of the device under test.